# k-nearest neighbor algorithm

https://towardsdatascience.com/getting-acquainted-with-k-nearest-neighbors-ba0a9ecf354f

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
import pandas as pd
N=1000

df1 = 2*pd.DataFrame(np.random.randn(N, 3), columns=['A', 'B','C'])
df2 = 20+10*pd.DataFrame(np.random.randn(N, 3), columns=['A', 'B','C'])

Data=np.concatenate((df1, df2), axis=0)
Data[:,2]=1
Data[0:N,2]=0

import matplotlib.pyplot as plt

s='b'*N+'r'*N
plt.scatter(Data[:,0],Data[:,1], s=3,alpha=1,c=list(s))



# Task 1

Please, write(without using any additional library) a function 
```nearest(x,k,Data)```
which return a predicted class for the new data piont ```x``` based on ```k```-nearest neighbor algorithm for dataset Data.

first approach 

In [ ]:

# musimy znaleźć wszystkie najbliższe punkty w tabeli i z opdowiednimi wagami je potraktować czyli:

Data = pd.DataFrame(Data, columns=['X', 'Y', 'G'])
# data - 1 grupa jedna 0 grupa druga in c 
def nearest(x, k, Data):
    '''x - point, k number of searched points
    weight is 1/distance '''
    # finding weights for given points 
    Data['weight'] = Data.apply(lambda row: np.sqrt((row['X'] - x[0])**2 + (row['Y'] - x[1])**2)**(-1), axis=1)
    # sorting by highest weights and taking k first rows
    sorted_data = Data.sort_values(by='weight', ascending=False).head(k)
    # applying values, based on group 
    sorted_data['score'] = sorted_data.apply(lambda row: (-1)**row['G'] * row['weight'], axis=1)
    # deciding which group to add 
    value = sorted_data['score'].sum()
    # optional adding to Data
    return value
    

print(nearest((-13, -24), 15, Data.copy()))
print(nearest((3.031840691297445, 3.4090250843027072), 15, Data.copy()))


# Task 2

Use the function meshgrid to plot prediction for each piont. See the result below. Please decide which k is the best, or at least 'good enough'.

attempt with meshgrid 

ad1 
we need to calculate distance for every point, we will use values from 'X' 'Y' labels respectively

In [ ]:
# settings
freq = 100
margin = 1
k = 5
#  finding biggest values 
A_min, A_max = np.floor(Data['X'].min()), np.ceil(Data['X'].max())
B_min, B_max = np.floor(Data['Y'].min()), np.ceil(Data['Y'].max())
# creating linear space
A = np.linspace(A_min - margin, A_max + margin, freq)
B = np.linspace(B_min - margin, B_max + margin, freq) 

# creating meshgrid 
xg, yg = np.meshgrid(A, B)

In [ ]:
# without selecting k points (calculating all of them)
Data.apply(lambda row: 
    ((-1)**row['G']*np.sqrt((row['X'] - xg)**2 + (row['Y'] - yg)**2))**-1, 
axis=1).sum()

selecting k rows method 

problems
- division by zero - getting infninity values
    - solution: by creating condition using using where function
- optimalization with looping for each element in xg yg matrix 
    - k value does not matter in complexity, only size of xg matrix !!!! O(n^2) and so more we are not using dedicasted @ operator for matrix operations 
    - we use one opeartor for x range and than summing it 
- drawing, how to draw them better (some options)

In [ ]:
# creating end result matrix
result = np.zeros_like(xg)
# iterating through every point of xg and yg:
for i in range(xg.shape[0]):
    for j in range(xg.shape[1]):
        # selecting point 
        x_grid, y_grid = xg[i, j], yg[i, j]
        # computing distances from selected point to all points in data
        distances = np.sqrt((Data['X'] - x_grid)**2 + (Data['Y'] - y_grid)**2)
        # finding nearest k indixes 
        idx = np.argsort((distances))[:k]
        # calculating value
        value = sum(
            (-1)**Data.loc[idx, 'G'] / distances[idx] 
        )
        value = 1 if value < 0 else 0
        result[i, j] = value 


In [ ]:
result_01 = result
plt.figure(figsize=(8, 6))
plt.contourf(xg, yg, result, alpha=0.3, cmap='coolwarm')  # Decision boundaries
plt.scatter(Data['X'], Data['Y'], c=Data['G'], edgecolors='k', cmap='coolwarm')  # Data points
plt.xlabel('X')
plt.ylabel('Y')
plt.title('k-NN Decision Boundary')
plt.colorbar(label="Class")
plt.show()

# Task 3

Read about ```train_test_split``` function (```sklearn.model_selection```). Split dataset into two parts: train and test set.
Check on test set how many times your function(trained on train set) made a mistake.

Do the same by using ```confusion_matrix``` (```sklearn.metrics```).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(Data[['X', 'Y']], Data['G'], random_state=5, test_size=0.2) # test sieze is end result

# tutaj po prostu robimy tą metode z drugiego zdania tylko na wybrakowanym data setcie


# Task 4

Repeat tasks 2 and 3, by using ```KNeighborsClassifier``` function (sklearn). What other parameters do we have?

notes on KNeighborsClassifier parameters:
- n_neighbors = k
- weights
    - allows ous to chose from weight, important are:
        - 'uniform' (default)
            - distance does not matter
        - 'distance'
            - 1/distance 
        - other
            it takes callable function which works on array of distances !!!
- metric
    - 'minkowski' defautl (can introduce different )

In [ ]:
from sklearn.neighbors import KNeighborsClassifier 

### Task 2 

In [ ]:
X = Data[['X', 'Y']]
y = Data['G']
# we are fitting our model
knn = KNeighborsClassifier(5, weights='distance')
knn.fit(X, y)
# preparing grids
X_test = np.column_stack((xg.ravel(), yg.ravel()))
y_pred = knn.predict(X_test)
# reshaping predictions 
y_pred_grid = y_pred.reshape(xg.shape)

In [ ]:
plt.figure(figsize=(8, 6))
plt.contourf(xg, yg, y_pred_grid, alpha=0.3, cmap='coolwarm')  # Decision boundaries
plt.scatter(Data['X'], Data['Y'], c=Data['G'], edgecolors='k', cmap='coolwarm')  # Data points
plt.xlabel('X')
plt.ylabel('Y')
plt.title('k-NN Decision Boundary')
plt.colorbar(label="Class")
plt.show()

### Task 3

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(Data[['X', 'Y']], Data['G'], 
                                                    random_state=5, test_size=0.2) # test sieze is end result
# creating knn model 
knn = KNeighborsClassifier(5)
# fitting to test data
knn.fit(X_train, y_train)
# predict score
y_pred = knn.predict(X_test)
# confusion matrix - it tells about errors of first and second type
cm = confusion_matrix(y_test, y_pred)
# testing score
knn.score(X_test, y_test), cm

### remarks 

k number:
- we can not chose k which is divisible by number of classes !! we can get points which are not classified 